# Detección de ROI facial y normalización de pose

Segunda etapa del preprocesamiento. Toma los clips de 5–7 s que produce
`01_preparar_dataset_equinos.ipynb` y genera frames **recortados a la cara del equino y alineados**,
de modo que el ojo y el hocico caigan siempre en la misma posición.

Cubre el **objetivo específico 2** del plan de trabajo: detección de la región de interés y
alineación de los rostros a lo largo de los clips.

## Por qué hace falta

1. **El fondo predice la clase.** Los 3 videos de Disgustado son interior de box; Interesado y
   Sereno son casi todos exterior. Un modelo entrenado sobre el frame completo puede separar
   Disgustado sin mirar la cara ni una vez.
2. **La pose de la cabeza varía sin control**, de frontal completo a perfil total. Comparar
   microexpresiones entre poses tan distintas no tiene sentido geométrico.

## Requisitos previos

Haber ejecutado `01_preparar_dataset_equinos.ipynb` hasta el Bloque 2, que deja
`inventario_clips.csv` y la carpeta `clips_equinos/`.

**Usar entorno con GPU** (Entorno de ejecución → Cambiar tipo de entorno → GPU).

## Estructura

| Bloque | Pasos | Qué hace |
|---|---|---|
| 1 | 1–4 | Configuración, funciones geométricas e inventario |
| 2 | 5–8 | Detección del equino y suavizado temporal |
| 3 | 9–11 | Keypoints faciales → `keypoints_faciales.csv` |
| 4 | 12–13 | ROI, alineación y recorte |
| 5 | 14–17 | Control de calidad y decisión |

## Paso 0 — Arranque: montar Drive y cargar el paquete `equinos`

In [ ]:
# ============================================================
# ARRANQUE — persistencia en Google Drive
# ============================================================
# Esta celda es autocontenida a propósito: corre ANTES de que el paquete `equinos` esté
# disponible, así que no puede importarlo.
#
# Resuelve en cascada de dónde sale el código:
#   1. de la copia en Drive (RAIZ/codigo/equinos)
#   2. clonando el repo, si hay un GITHUB_TOKEN en los secrets de Colab
#   3. si no, falla con instrucciones concretas
#
# El repo es privado, por eso un `git clone` sin credenciales no alcanza. El token va en los
# secrets de Colab (panel izquierdo, icono de llave) y NUNCA dentro del notebook: lo que se
# escribe en una celda queda en el archivo y se comparte con él.

import shutil
import subprocess
import sys
from pathlib import Path

NOMBRE_PROYECTO = "tesina_equinos"
REPO = "github.com/sergioyanez/Tesina-UGR.git"

try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if EN_COLAB:
    from google.colab import drive

    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")

    CODIGO = Path("/content/drive/MyDrive") / NOMBRE_PROYECTO / "codigo"
    CODIGO.mkdir(parents=True, exist_ok=True)

    if not (CODIGO / "equinos").is_dir():
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN")
        except Exception:
            token = None

        if not token:
            raise SystemExit(
                f"Falta el paquete `equinos` en {CODIGO}.\n"
                "  Opción A: cargá un secret GITHUB_TOKEN en Colab (panel izquierdo, icono de llave).\n"
                f"  Opción B: copiá a mano la carpeta `equinos/` del repo dentro de {CODIGO}."
            )

        clon = Path("/content/_repo_tesina")
        shutil.rmtree(clon, ignore_errors=True)
        resultado = subprocess.run(
            ["git", "clone", "--depth", "1", f"https://{token}@{REPO}", str(clon)],
            capture_output=True, text=True,
        )
        if resultado.returncode != 0:
            # No se imprime el comando ni stderr: contienen el token.
            raise SystemExit("Falló el clonado. Verificá que GITHUB_TOKEN tenga permiso de lectura del repo.")

        shutil.copytree(clon / "equinos", CODIGO / "equinos", dirs_exist_ok=True)
        print(f"Paquete `equinos` copiado a {CODIGO}")

    sys.path.insert(0, str(CODIGO))
else:
    sys.path.insert(0, str(Path.cwd()))

from equinos import cache, checkpoint, etapas, rutas

RAIZ = rutas.inicializar()


## Paso 0a — Parámetros de la etapa

In [ ]:
# Parámetros del preprocesamiento facial.
#
# Papermill inyecta una celda justo debajo de ésta cuando el notebook se ejecuta desde
# 00_pipeline.ipynb. Ejecutado a mano, valen estos valores por defecto.
# Están todos juntos acá para poder barrerlos sin tocar el resto del notebook.

CONF_MINIMA_DETECCION = 0.50   # confianza mínima para aceptar una caja de equino
CONF_MINIMA_KEYPOINT  = 0.50   # confianza mínima por keypoint facial
VENTANA_SUAVIZADO     = 5      # frames de la mediana móvil sobre las cajas
MARGEN_ROI            = 0.45   # aire alrededor de la cara, como fracción del lado
LADO_RECORTE          = 224    # entrada estándar de los backbones preentrenados
FRAC_OJO              = 0.34   # posición vertical canónica del ojo en el recorte
FRAC_HOCICO           = 0.74   # posición vertical canónica del hocico

# Umbral por debajo del cual el pipeline automático se considera no utilizable
COBERTURA_MINIMA_ACEPTABLE = 0.70

# True descarta lo ya hecho y re-ejecuta la etapa desde cero.
LIMPIAR_ETAPA = False


## Bloque 1: Configuración

Paso 1 — Instalar dependencias

Paso 2 — Constantes

Paso 3 — Funciones geométricas

Paso 4 — Cargar inventario de clips

## Paso 1 — Instalar dependencias


In [ ]:
# ============================================================
# DEPENDENCIAS
# ============================================================
# TODAS las instalaciones van acá, antes de importar nada del stack científico, y en UN SOLO
# comando para que pip resuelva un conjunto consistente en vez de pisarse a sí mismo.
#
# El problema concreto: DeepLabCut fija numpy<2, y Colab viene con scipy y pandas compilados
# contra numpy 2. Instalarlo a secas baja numpy a 1.x y rompe scipy con
# "No module named 'numpy.strings'". Por eso scipy va pinneado a la última serie que convive
# con numpy 1.x.
#
# Y por eso la instalación va en la PRIMERA celda: si numpy ya está importado en memoria cuando
# pip lo reemplaza en disco, todo revienta con "numpy.dtype size changed", y bajo papermill no
# hay forma de reiniciar el kernel para recuperarse.

!pip install -q deeplabcut "scipy<1.14" ultralytics opencv-python pandas numpy tqdm matplotlib

# Verificación temprana del entorno. Sin esto, una incompatibilidad de versiones se manifiesta
# 60 líneas de traceback más abajo y dos minutos después, en `import deeplabcut`.
import numpy
import scipy
import pandas
import torch

print(f"numpy {numpy.__version__} | scipy {scipy.__version__} | pandas {pandas.__version__}")
print("GPU disponible:", torch.cuda.is_available())

if not torch.cuda.is_available():
    print("ADVERTENCIA: sin GPU el procesamiento de 1472 frames va a ser muy lento.")
    print("Entorno de ejecución -> Cambiar tipo de entorno -> GPU")


## Paso 2 — Rutas de la etapa

In [ ]:
# ============================================================
# RUTAS
# ============================================================
# Todas cuelgan de la raíz de proyecto persistente, que resolvió el Paso 0. Los frames
# alineados van al área de trabajo local y se persisten empaquetados (Paso 13): son ~1472
# archivos chicos, y Drive montado tiene una latencia por archivo que domina el tiempo total.

resultados_dir = rutas.carpeta("resultados")
clips_dir      = rutas.carpeta("clips")
alineados_dir  = rutas.dir_trabajo("frames_alineados")

inventario_clips_path = resultados_dir / "inventario_clips.csv"

if LIMPIAR_ETAPA:
    etapas.limpiar("roi_normalizacion")

print("Clips de entrada:", clips_dir)
print("Frames alineados:", alineados_dir)


## Paso 3 — Funciones geométricas

Estas funciones son **puras**: no dependen de ningún modelo ni de GPU, y están probadas por
separado con casos sintéticos, incluidos los degenerados. Son la parte del pipeline donde se
esconden los errores silenciosos, así que conviene tenerlas aisladas y verificadas.

In [ ]:
import numpy as np

# ============================================================
# SUAVIZADO TEMPORAL DE CAJAS
# ============================================================

def suavizar_cajas(cajas, ventana=5):
    """
    Aplica una mediana movil a una secuencia de cajas [x1, y1, x2, y2].

    El caballo y la camara se mueven despacio, asi que la caja real varia
    de forma suave entre frames. La mediana elimina los saltos bruscos de
    una deteccion aislada mal ubicada sin correr los bordes, cosa que un
    promedio si haria.

    Las entradas None (frames sin deteccion) se ignoran para calcular la
    mediana, pero se rellenan con el valor suavizado de su posicion.
    """
    n = len(cajas)
    if n == 0:
        return []

    salida = []
    mitad = max(1, ventana // 2)

    for i in range(n):
        desde = max(0, i - mitad)
        hasta = min(n, i + mitad + 1)
        vecinas = [c for c in cajas[desde:hasta] if c is not None]

        if not vecinas:
            salida.append(None)
            continue

        salida.append(tuple(np.median(np.array(vecinas, dtype=float), axis=0)))

    return salida


def rellenar_faltantes(cajas):
    """
    Rellena las posiciones sin deteccion con la caja valida mas cercana.

    Devuelve (cajas_rellenadas, cantidad_rellenada). Si no hay ninguna
    deteccion valida devuelve la lista original y -1.
    """
    validas = [i for i, c in enumerate(cajas) if c is not None]

    if not validas:
        return list(cajas), -1

    salida = list(cajas)
    rellenadas = 0

    for i in range(len(cajas)):
        if cajas[i] is not None:
            continue
        j = min(validas, key=lambda k: abs(k - i))
        salida[i] = cajas[j]
        rellenadas += 1

    return salida, rellenadas


# ============================================================
# TRANSFORMADA DE SIMILITUD A PARTIR DE DOS PUNTOS
# ============================================================

def similitud_desde_dos_puntos(p1, p2, q1, q2):
    """
    Construye la matriz afin 2x3 de la similitud que lleva p1->q1 y p2->q2.

    Dos correspondencias determinan exactamente una similitud (4 grados de
    libertad: escala, rotacion, y traslacion en x e y). Es la misma idea que
    se usa para alinear rostros humanos por la linea entre los ojos; aca la
    referencia es el eje ojo-hocico, que define la orientacion de la cabeza
    tanto de frente como de perfil.

    Devuelve None si los puntos de origen coinciden (no hay eje que alinear).
    """
    p1 = np.asarray(p1, dtype=float)
    p2 = np.asarray(p2, dtype=float)
    q1 = np.asarray(q1, dtype=float)
    q2 = np.asarray(q2, dtype=float)

    v = p2 - p1
    w = q2 - q1

    norma_v = np.hypot(*v)
    if norma_v < 1e-8:
        return None

    escala = np.hypot(*w) / norma_v
    angulo = np.arctan2(w[1], w[0]) - np.arctan2(v[1], v[0])

    cos_a = np.cos(angulo) * escala
    sin_a = np.sin(angulo) * escala

    rotacion = np.array([[cos_a, -sin_a],
                         [sin_a,  cos_a]])

    traslacion = q1 - rotacion @ p1

    return np.array([[rotacion[0, 0], rotacion[0, 1], traslacion[0]],
                     [rotacion[1, 0], rotacion[1, 1], traslacion[1]]])


def destino_canonico(lado, frac_ojo=0.34, frac_hocico=0.74):
    """
    Posiciones de destino del ojo y del hocico en el recorte normalizado.

    Ambos quedan sobre el eje vertical central, separados por
    (frac_hocico - frac_ojo) * lado. Asi todos los recortes salen con la
    cabeza en la misma orientacion y escala, sin importar si el video la
    captó de frente o de perfil.
    """
    return (lado * 0.5, lado * frac_ojo), (lado * 0.5, lado * frac_hocico)


# ============================================================
# REGION DE INTERES A PARTIR DE KEYPOINTS
# ============================================================

def roi_desde_keypoints(puntos, margen=0.45, forma=None):
    """
    Caja cuadrada que cubre los keypoints faciales con un margen relativo.

    Se fuerza cuadrada para que el reescalado posterior no deforme la cara.
    'margen' es la fraccion del lado que se agrega a cada costado.

    Si se pasa 'forma' (alto, ancho), la caja se recorta al borde de la imagen.
    """
    pts = np.asarray([p for p in puntos if p is not None], dtype=float)
    if len(pts) == 0:
        return None

    x1, y1 = pts.min(axis=0)
    x2, y2 = pts.max(axis=0)

    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
    lado = max(x2 - x1, y2 - y1) * (1 + 2 * margen)
    lado = max(lado, 1.0)

    caja = [cx - lado / 2, cy - lado / 2, cx + lado / 2, cy + lado / 2]

    if forma is not None:
        alto, ancho = forma[:2]
        caja[0] = max(0.0, caja[0])
        caja[1] = max(0.0, caja[1])
        caja[2] = min(float(ancho), caja[2])
        caja[3] = min(float(alto), caja[3])

    return tuple(caja)

## Paso 4 — Cargar el inventario de clips

In [ ]:
import pandas as pd

if not inventario_clips_path.exists():
    raise SystemExit(
        f"No existe {inventario_clips_path}.\n"
        "Hay que ejecutar antes 01_preparar_dataset_equinos.ipynb, que lo produce.\n"
        "Desde 00_pipeline.ipynb esto se encadena solo."
    )

df_clips = pd.read_csv(inventario_clips_path)

# Las rutas del CSV son relativas a la raíz de proyecto: se resuelven contra la raíz vigente,
# que puede ser otra sesión de Colab o la máquina local.
df_clips["existe"] = df_clips["ruta_clip"].apply(lambda r: rutas.resolver(r).exists())
faltantes = df_clips[~df_clips["existe"]]

print("Clips en el inventario:", len(df_clips))
print("Clips faltantes en disco:", len(faltantes))

if len(faltantes) > 0:
    print("\nEl inventario referencia clips que no están en la raíz de proyecto.")
    print(faltantes[["clase", "clip"]].head(10))

df_clips = df_clips[df_clips["existe"]].reset_index(drop=True)

print("\nClips por clase:")
print(df_clips["clase"].value_counts())
print("\nCaballos distintos:", df_clips["id_caballo"].nunique())
df_clips.head()


## Bloque 2: Detección del equino

Paso 5 — Cargar el detector y localizar la clase `horse`

Paso 6 — Detectar en cada frame

Paso 7 — Suavizado temporal y relleno de huecos

Paso 8 — Cobertura de la detección

Se usa un detector preentrenado en COCO, que ya incluye la clase `horse` entre sus 80 categorías.
**No requiere ninguna anotación manual.** La caja de cuerpo completo no es el objetivo final, pero
acota el espacio de búsqueda y sirve de respaldo cuando fallan los keypoints faciales.

## Paso 5 — Cargar el detector y localizar la clase `horse`

In [ ]:
from ultralytics import YOLO

# ============================================================
# DETECTOR PREENTRENADO EN COCO
# ============================================================

detector = YOLO("yolo11x.pt")   # se descarga automáticamente la primera vez

# El índice de la clase NO se hardcodea: se busca por nombre en el propio
# modelo. Si en algún momento se cambia de detector, esto sigue funcionando.
nombres = detector.names
CLASE_EQUINO = None

for indice, nombre in nombres.items():
    if nombre.lower() in ("horse", "caballo"):
        CLASE_EQUINO = indice
        break

if CLASE_EQUINO is None:
    raise RuntimeError(
        f"El modelo no expone una clase de equino. Clases disponibles: {list(nombres.values())}"
    )

print(f"Clase de equino encontrada: índice {CLASE_EQUINO} -> '{nombres[CLASE_EQUINO]}'")
print("Total de clases del modelo:", len(nombres))

## Paso 6 — Detectar el equino en cada frame

Se procesan los mismos 16 frames por clip que usa el Bloque 3, para que ambos datasets —el crudo y
el alineado— sean comparables frame a frame.

In [ ]:
import cv2
import numpy as np
from tqdm import tqdm

NUM_FRAMES = 16

def indices_uniformes(total_frames, cantidad):
    """Mismos índices que usa el Bloque 3, para poder comparar ambos datasets."""
    if total_frames <= 0:
        return []
    return np.linspace(0, total_frames - 1, cantidad).astype(int).tolist()


detecciones = []   # una fila por frame

for _, fila in tqdm(df_clips.iterrows(), total=len(df_clips), desc="Detectando"):
    ruta_clip = rutas.resolver(fila["ruta_clip"])
    cap = cv2.VideoCapture(str(ruta_clip))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    indices = indices_uniformes(total, NUM_FRAMES)

    for orden, idx in enumerate(indices):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ok, frame = cap.read()

        registro = {
            "clase": fila["clase"],
            "id_caballo": fila["id_caballo"],
            "video_origen": fila["video_origen"],
            "clip": fila["clip"],
            "ruta_clip": fila["ruta_clip"],
            "orden": orden,
            "frame_idx": int(idx),
            "caja": None,
            "confianza": 0.0,
            "n_detecciones": 0,
        }

        if not ok:
            detecciones.append(registro)
            continue

        resultado = detector.predict(
            frame, classes=[CLASE_EQUINO], conf=CONF_MINIMA_DETECCION, verbose=False
        )[0]

        cajas = resultado.boxes
        registro["n_detecciones"] = 0 if cajas is None else len(cajas)

        if cajas is not None and len(cajas) > 0:
            # Cuando hay más de un equino, se conserva el de mayor confianza
            confianzas = cajas.conf.cpu().numpy()
            mejor = int(np.argmax(confianzas))
            registro["caja"] = tuple(cajas.xyxy.cpu().numpy()[mejor].tolist())
            registro["confianza"] = float(confianzas[mejor])

        detecciones.append(registro)

    cap.release()

df_det = pd.DataFrame(detecciones)

print("Frames procesados:", len(df_det))
print("Frames con detección:", int(df_det["caja"].notna().sum()))
print("Frames con más de un equino:", int((df_det["n_detecciones"] > 1).sum()))
df_det.head()

## Paso 7 — Suavizado temporal y relleno de huecos

La mediana móvil se aplica **por clip**, nunca cruzando clips distintos: dos clips consecutivos del
mismo video pueden tener al animal en posiciones muy diferentes.

In [ ]:
# ============================================================
# SUAVIZADO POR CLIP
# ============================================================

cajas_suaves = {}
relleno_por_clip = {}
clips_sin_deteccion = []

for clip, grupo in df_det.groupby("clip", sort=False):
    grupo = grupo.sort_values("orden")
    cajas = [c if isinstance(c, tuple) else None for c in grupo["caja"]]

    suaves = suavizar_cajas(cajas, ventana=VENTANA_SUAVIZADO)
    rellenas, n_rellenadas = rellenar_faltantes(suaves)

    if n_rellenadas == -1:
        clips_sin_deteccion.append(clip)
        continue

    relleno_por_clip[clip] = n_rellenadas
    for orden, caja in zip(grupo["orden"], rellenas):
        cajas_suaves[(clip, orden)] = caja

df_det["caja_suave"] = [
    cajas_suaves.get((c, o)) for c, o in zip(df_det["clip"], df_det["orden"])
]
df_det["caja_rellenada"] = df_det["caja"].isna() & df_det["caja_suave"].notna()

print("Clips sin ninguna detección (excluidos):", len(clips_sin_deteccion))
for c in clips_sin_deteccion:
    print("   ", c)

print("\nFrames rellenados:", int(df_det["caja_rellenada"].sum()))
print("Frames con caja utilizable:", int(df_det["caja_suave"].notna().sum()))

## Paso 8 — Cobertura de la detección

In [ ]:
# ============================================================
# COBERTURA POR CLASE Y POR CABALLO
# ============================================================

df_det["con_caja"] = df_det["caja_suave"].notna()

print("Cobertura global de detección: {:.1%}".format(df_det["con_caja"].mean()))

print("\nPor clase:")
print(df_det.groupby("clase")["con_caja"].agg(["mean", "count"]).rename(
    columns={"mean": "cobertura", "count": "frames"}))

print("\nPor caballo:")
cob_caballo = df_det.groupby("id_caballo")["con_caja"].mean().sort_values()
print(cob_caballo)

bajos = cob_caballo[cob_caballo < COBERTURA_MINIMA_ACEPTABLE]
if len(bajos) > 0:
    print("\nADVERTENCIA: caballos por debajo del umbral aceptable "
          f"({COBERTURA_MINIMA_ACEPTABLE:.0%}):")
    print(bajos)
else:
    print(f"\nTodos los caballos superan el {COBERTURA_MINIMA_ACEPTABLE:.0%} de cobertura.")

## Bloque 3: Keypoints faciales

Paso 9 — Cargar el modelo de pose

Paso 10 — Descubrir el esquema, seleccionar el par de referencia y extraer los keypoints

Paso 11 — Cobertura de keypoints

Se usa **DeepLabCut SuperAnimal-Quadruped**: estima keypoints en cuadrúpedos sin entrenar ni anotar
nada. Verificado en Colab sobre un clip real (~35 s de inferencia por clip en GPU, más ~10 s de
video etiquetado que después se descarta).

### Contrato de `keypoints_faciales.csv`

| Columna | Significado |
|---|---|
| `clip` | Nombre del clip |
| `orden` | Índice del frame dentro del clip (0 a 15) |
| `x_ojo`, `y_ojo` | Coordenadas del ojo en píxeles de la imagen original |
| `conf_ojo` | Confianza del keypoint ocular |
| `x_hocico`, `y_hocico` | Coordenadas del hocico en píxeles de la imagen original |
| `conf_hocico` | Confianza del keypoint del hocico |

Todo lo posterior consume **únicamente** este CSV. Cualquier fuente que lo produzca sirve: el modelo
automático, otro modelo, o anotación manual.

## Paso 9 — Cargar el modelo de pose

Se usa **DeepLabCut SuperAnimal-Quadruped**, que estima keypoints en cuadrúpedos sin entrenar ni
anotar nada (Mathis et al., 2018, citado en la bibliografía del plan de trabajo).

El esquema de keypoints **no se adivina desde un archivo de configuración**: se lee del propio
resultado de la inferencia, que es la única fuente confiable. Por eso este paso sólo carga la
librería y define la función de inferencia; el esquema aparece en el Paso 10.

In [ ]:
# ============================================================
# MODELO DE POSE PARA CUADRÚPEDOS
# ============================================================
# DeepLabCut ya quedó instalado en el Paso 1, junto al resto de las dependencias.

import deeplabcut
import pandas as pd
import numpy as np
from pathlib import Path

print("DeepLabCut:", getattr(deeplabcut, "__version__", "?"))

# Archivos auxiliares que genera DeepLabCut junto a cada clip. Se borran
# después de extraer los keypoints: el .h5 y el .pickle son intermedios y el
# video etiquetado pesa casi como el clip original. Sin limpiarlos, se duplica
# el espacio ocupado y se ensucia clips_equinos/, que otros pasos recorren.
SUFIJOS_AUXILIARES = (".h5", ".pickle", "_labeled_before_adapt.mp4", "_labeled.mp4")


def inferir_keypoints_clip(ruta_clip):
    """
    Corre la inferencia sobre un clip.

    Devuelve el DataFrame de poses (filas = frames, columnas MultiIndex
    scorer/individuals/bodyparts/coords) o None si la inferencia falla.
    """
    ruta_clip = str(ruta_clip)
    try:
        resultado = deeplabcut.video_inference_superanimal(
            [ruta_clip],
            superanimal_name="superanimal_quadruped",
            model_name="hrnet_w32",
            detector_name="fasterrcnn_resnet50_fpn_v2",
        )
    except Exception as e:
        print(f"  fallo en {Path(ruta_clip).name}: {type(e).__name__}: {e}")
        return None

    # El resultado es un dict indexado por la ruta del video. Se toma el único
    # valor en lugar de reconstruir la clave, que puede venir normalizada.
    if not isinstance(resultado, dict) or not resultado:
        print(f"  resultado inesperado en {Path(ruta_clip).name}: {type(resultado)}")
        return None

    return next(iter(resultado.values()))


def limpiar_auxiliares(ruta_clip):
    """Borra los archivos que DeepLabCut deja junto al clip."""
    carpeta = Path(ruta_clip).parent
    raiz = Path(ruta_clip).stem
    borrados = 0
    for archivo in carpeta.glob(f"{raiz}_*"):
        if archivo.suffix in (".h5", ".pickle") or archivo.name.endswith(".mp4"):
            if archivo.name != Path(ruta_clip).name:
                archivo.unlink()
                borrados += 1
    return borrados


print("Función de inferencia lista.")

### Descubrir el esquema de keypoints

Se corre la inferencia sobre **un solo clip** y se lee la lista de keypoints directamente de las
columnas del DataFrame. Es la fuente autoritativa: cualquier otra vía depende de rutas internas de
la librería que cambian entre versiones.

In [ ]:
# ============================================================
# ESQUEMA REAL DEL MODELO
# ============================================================

clip_prueba = rutas.resolver(df_clips.iloc[0]["ruta_clip"])
print("Inferencia de prueba sobre:", Path(clip_prueba).name)

pose_prueba = inferir_keypoints_clip(clip_prueba)

if pose_prueba is None:
    raise RuntimeError("La inferencia de prueba falló. Revisar la instalación de DeepLabCut.")

print("\nFilas (frames):", len(pose_prueba))
print("Niveles de columnas:", pose_prueba.columns.names)

SCORER      = pose_prueba.columns.get_level_values("scorer")[0]
INDIVIDUOS  = list(dict.fromkeys(pose_prueba.columns.get_level_values("individuals")))
BODYPARTS   = list(dict.fromkeys(pose_prueba.columns.get_level_values("bodyparts")))
COORDS      = list(dict.fromkeys(pose_prueba.columns.get_level_values("coords")))

print("Scorer:", SCORER)
print("Individuos:", INDIVIDUOS)
print("Coords:", COORDS)
print(f"\nKeypoints disponibles ({len(BODYPARTS)}):")
for i, b in enumerate(BODYPARTS):
    print(f"  {i:3d}  {b}")

## Paso 10 — Seleccionar el par de referencia y extraer los keypoints

### Por qué se elige el ojo por frame y no uno fijo

En perfil cerrado **sólo se ve un ojo**, y cuál es depende de hacia dónde mire el animal. Fijar
`left_eye` de antemano perdería todos los frames en los que el caballo mira para el otro lado.

Por eso, en cada frame, se toma el ojo con mayor `likelihood`. Es la diferencia entre aprovechar
todos los frames o la mitad.

Para el hocico se usa `nose`, que es la punta del hocico y el extremo del eje longitudinal de la
cabeza. Junto con el ojo define la orientación tanto de frente como de perfil.

In [ ]:
# ============================================================
# SELECCIÓN DEL PAR OJO / HOCICO
# ============================================================
# Dos puntos determinan exactamente una similitud (escala, rotación y
# traslación), así que no hace falta ajuste por mínimos cuadrados.

BODYPARTS_OJO = [b for b in BODYPARTS if "eye" in b.lower()]

CANDIDATOS_HOCICO = ["nose", "snout", "muzzle", "upper_jaw", "mouth"]
BODYPART_HOCICO = None
for cand in CANDIDATOS_HOCICO:
    for b in BODYPARTS:
        if b.lower() == cand:
            BODYPART_HOCICO = b
            break
    if BODYPART_HOCICO:
        break

print("Keypoints oculares (se elige el más confiable en cada frame):", BODYPARTS_OJO)
print("Keypoint del hocico:", BODYPART_HOCICO)

if not BODYPARTS_OJO or BODYPART_HOCICO is None:
    raise RuntimeError(
        "No se pudo seleccionar el par de referencia. "
        f"Elegir a mano de esta lista: {BODYPARTS}"
    )

### Extracción a formato canónico

Todo lo que sigue en el notebook consume **únicamente** `keypoints_faciales.csv`. Si en algún
momento hay que reemplazar el modelo de pose por anotación manual, este es el único punto a tocar.

In [ ]:
# ============================================================
# EXTRACCIÓN DE KEYPOINTS A FORMATO CANÓNICO
# ============================================================

def serie_keypoint(pose, bodypart, individuo):
    """Devuelve (x, y, likelihood) como arrays, para un keypoint y un individuo."""
    sub = pose.xs((SCORER, individuo, bodypart), axis=1,
                  level=("scorer", "individuals", "bodyparts"))
    return (sub["x"].to_numpy(dtype=float),
            sub["y"].to_numpy(dtype=float),
            sub["likelihood"].to_numpy(dtype=float))


def elegir_individuo(pose):
    """Con varios animales detectados, se queda con el de mayor confianza media."""
    if len(INDIVIDUOS) == 1:
        return INDIVIDUOS[0]
    mejor, mejor_conf = INDIVIDUOS[0], -1.0
    for ind in INDIVIDUOS:
        confs = [serie_keypoint(pose, b, ind)[2] for b in BODYPARTS]
        media = float(np.nanmean(np.concatenate(confs))) if confs else -1.0
        if media > mejor_conf:
            mejor, mejor_conf = ind, media
    return mejor


def extraer_par_facial(pose, ordenes_e_indices):
    """
    Traduce el DataFrame de poses a filas del contrato documentado.

    Por cada frame toma el ojo de mayor likelihood entre los disponibles: en
    perfil sólo uno es visible, y cuál depende de hacia dónde mire el animal.
    """
    individuo = elegir_individuo(pose)
    n_filas = len(pose)

    ojos = [serie_keypoint(pose, b, individuo) for b in BODYPARTS_OJO]
    hx, hy, hc = serie_keypoint(pose, BODYPART_HOCICO, individuo)

    filas = []
    for orden, frame_idx in ordenes_e_indices:
        # CAP_PROP_FRAME_COUNT puede sobreestimar; se acota al rango real
        idx = int(min(max(frame_idx, 0), n_filas - 1))

        # Ojo más confiable de este frame
        mejor_x = mejor_y = np.nan
        mejor_c = -1.0
        for ox, oy, oc in ojos:
            if oc[idx] > mejor_c:
                mejor_x, mejor_y, mejor_c = ox[idx], oy[idx], oc[idx]

        filas.append({
            "orden": int(orden),
            "x_ojo": float(mejor_x), "y_ojo": float(mejor_y),
            "conf_ojo": float(max(mejor_c, 0.0)),
            "x_hocico": float(hx[idx]), "y_hocico": float(hy[idx]),
            "conf_hocico": float(hc[idx]),
        })

    return filas


# --- Verificación sobre el clip de prueba antes de lanzar los 92 ---
grupo_prueba = df_det[df_det["clip"] == df_clips.iloc[0]["clip"]].sort_values("orden")
muestra = extraer_par_facial(pose_prueba,
                             list(zip(grupo_prueba["orden"], grupo_prueba["frame_idx"])))

print("Extracción de prueba, primeros 3 frames:")
for f in muestra[:3]:
    print("  ", f)

conf_media = np.mean([f["conf_ojo"] for f in muestra])
print(f"\nConfianza media del ojo en el clip de prueba: {conf_media:.3f}")

In [ ]:
# ============================================================
# INFERENCIA SOBRE TODOS LOS CLIPS  (reanudable)
# ============================================================
# Aproximadamente 45 s por clip en GPU (detector + pose + video etiquetado), así que los 92
# llevan alrededor de una hora: tiempo de sobra para que Colab corte la sesión en el medio.
# Con checkpoint por clip, una interrupción cuesta menos de un minuto de trabajo.
#
# El clip se copia al disco local antes de inferir porque DeepLabCut escribe sus auxiliares
# (.h5, .pickle y un video etiquetado que pesa como el original) junto al archivo de entrada.
# Dejarlos caer en Drive sería lento y llenaría la cuenta de basura temporal.

import shutil

from tqdm import tqdm

# Esta etapa no depende de ningún umbral nuestro: DeepLabCut devuelve coordenadas y su
# likelihood, y el filtrado por CONF_MINIMA_KEYPOINT ocurre después, en el Paso 11. Por eso la
# lista de claves relevantes va vacía: barrer umbrales no obliga a reinferir la hora de cómputo.
CLAVES_KEYPOINTS = []

cp = checkpoint.abrir("roi_normalizacion", globals(), CLAVES_KEYPOINTS)

tmp_dir = rutas.dir_trabajo("clips_inferencia")
pendientes = cp.pendientes(df_clips["clip"].tolist())
print(f"Clips pendientes: {len(pendientes)} de {len(df_clips)}\n")

clips_fallidos = []
por_procesar = df_clips[df_clips["clip"].isin(pendientes)]

for _, fila in tqdm(por_procesar.iterrows(), total=len(por_procesar), desc="Keypoints"):
    local = tmp_dir / fila["clip"]
    shutil.copy(rutas.resolver(fila["ruta_clip"]), local)

    try:
        pose = inferir_keypoints_clip(local)

        if pose is None:
            clips_fallidos.append(fila["clip"])
            cp.marcar(fila["clip"])
            continue

        grupo = df_det[df_det["clip"] == fila["clip"]].sort_values("orden")
        ordenes_e_indices = list(zip(grupo["orden"], grupo["frame_idx"]))

        filas_clip = extraer_par_facial(pose, ordenes_e_indices)
        for f in filas_clip:
            f["clip"] = fila["clip"]

        # Primero las filas, después la marca: si el proceso muere en el medio, lo peor que
        # pasa es reprocesar un clip.
        cp.marcar(fila["clip"], filas_clip)
    finally:
        limpiar_auxiliares(local)
        local.unlink(missing_ok=True)

df_kp = cp.filas()

kp_csv_path = resultados_dir / "keypoints_faciales.csv"
df_kp.to_csv(kp_csv_path, index=False)

print("\nKeypoints guardados en:", kp_csv_path)
print("Filas:", len(df_kp))
print("Clips fallidos:", len(clips_fallidos), clips_fallidos)
df_kp.head()


## Paso 11 — Cobertura de keypoints

In [ ]:
# ============================================================
# ¿CUÁNTOS FRAMES TIENEN EL PAR OJO/HOCICO UTILIZABLE?
# ============================================================

df_kp["par_valido"] = (
    (df_kp["conf_ojo"] >= CONF_MINIMA_KEYPOINT) &
    (df_kp["conf_hocico"] >= CONF_MINIMA_KEYPOINT) &
    df_kp["x_ojo"].notna() & df_kp["x_hocico"].notna()
)

df_det = df_det.merge(df_kp, on=["clip", "orden"], how="left")
df_det["par_valido"] = df_det["par_valido"].fillna(False)

print("Cobertura global de keypoints: {:.1%}".format(df_det["par_valido"].mean()))

print("\nPor clase:")
print(df_det.groupby("clase")["par_valido"].agg(["mean", "count"]).rename(
    columns={"mean": "cobertura", "count": "frames"}))

print("\nPor caballo:")
cob_kp = df_det.groupby("id_caballo")["par_valido"].mean().sort_values()
print(cob_kp)

if df_det["par_valido"].mean() < COBERTURA_MINIMA_ACEPTABLE:
    print("\n" + "=" * 60)
    print("COBERTURA INSUFICIENTE")
    print("=" * 60)
    print("El pipeline automático de keypoints no alcanza el umbral aceptable.")
    print("Opciones, en este orden:")
    print("  1. Bajar CONF_MINIMA_KEYPOINT y volver a evaluar.")
    print("  2. Revisar que la extracción del Paso 10 corresponda a la estructura real.")
    print("  3. Anotar manualmente una ROI por clip (92 cajas) y generar")
    print("     keypoints_faciales.csv a mano con el mismo formato documentado.")

## Bloque 4: ROI y alineación

Paso 12 — Recortar y alinear

Paso 13 — Generar `dataset_frames_alineados.csv`

## Paso 12 — Recortar y alinear

Cada frame sigue uno de dos caminos:

- **Con par ojo/hocico válido** → transformada de similitud que los lleva a la posición canónica.
  El recorte sale alineado.
- **Sin par válido** → ROI cuadrada derivada de la caja de detección, sin alinear. Queda marcado
  con `alineado = False` para poder filtrarlo o analizarlo aparte más adelante.

Nunca se inventan coordenadas: si no hay dato, se registra la ausencia.

In [ ]:
# ============================================================
# RECORTE Y ALINEACIÓN
# ============================================================

dest_ojo, dest_hocico = destino_canonico(LADO_RECORTE, FRAC_OJO, FRAC_HOCICO)

registros_alineados = []
cache_frames = {}

for clip, grupo in tqdm(df_det.groupby("clip", sort=False), desc="Alineando"):
    grupo = grupo.sort_values("orden")
    ruta_clip = rutas.resolver(grupo.iloc[0]["ruta_clip"])

    cap = cv2.VideoCapture(str(ruta_clip))
    salida_dir = alineados_dir / grupo.iloc[0]["clase"] / Path(ruta_clip).stem
    salida_dir.mkdir(parents=True, exist_ok=True)

    for _, fila in grupo.iterrows():
        if fila["caja_suave"] is None:
            continue

        cap.set(cv2.CAP_PROP_POS_FRAMES, int(fila["frame_idx"]))
        ok, frame = cap.read()
        if not ok:
            continue

        alineado = False

        if fila["par_valido"]:
            # Camino principal: alineación por transformada de similitud
            M = similitud_desde_dos_puntos(
                (fila["x_ojo"], fila["y_ojo"]),
                (fila["x_hocico"], fila["y_hocico"]),
                dest_ojo, dest_hocico,
            )
            if M is not None:
                recorte = cv2.warpAffine(frame, M, (LADO_RECORTE, LADO_RECORTE))
                alineado = True

        if not alineado:
            # Degradación: ROI cuadrada a partir de la caja, sin alinear
            x1, y1, x2, y2 = fila["caja_suave"]
            roi = roi_desde_keypoints([(x1, y1), (x2, y2)],
                                      margen=0.0, forma=frame.shape)
            if roi is None:
                continue
            rx1, ry1, rx2, ry2 = [int(round(v)) for v in roi]
            if rx2 - rx1 < 2 or ry2 - ry1 < 2:
                continue
            recorte = cv2.resize(frame[ry1:ry2, rx1:rx2],
                                 (LADO_RECORTE, LADO_RECORTE))

        nombre = f"frame_{int(fila['orden']):03d}.jpg"
        ruta_salida = salida_dir / nombre
        cv2.imwrite(str(ruta_salida), recorte)

        registros_alineados.append({
            "clase": fila["clase"],
            "id_caballo": fila["id_caballo"],
            "video_origen": fila["video_origen"],
            "clip": fila["clip"],
            "frame": nombre,
            "ruta_frame": ruta_salida.relative_to(alineados_dir).as_posix(),
            "orden": int(fila["orden"]),
            "alineado": alineado,
            "caja_rellenada": bool(fila["caja_rellenada"]),
            "confianza_deteccion": float(fila["confianza"]),
        })

    cap.release()

df_alineados = pd.DataFrame(registros_alineados)

print("Frames generados:", len(df_alineados))
print("Alineados por keypoints: {} ({:.1%})".format(
    int(df_alineados["alineado"].sum()), df_alineados["alineado"].mean()))
df_alineados.head()

## Paso 13 — Guardar `dataset_frames_alineados.csv`

In [ ]:
alineados_csv_path = resultados_dir / "dataset_frames_alineados.csv"
df_alineados.to_csv(alineados_csv_path, index=False)

print("CSV guardado en:", alineados_csv_path)
print("\nColumnas:", list(df_alineados.columns))

# id_caballo tiene que sobrevivir hasta acá: es la clave de agrupamiento
# para particionar sin fuga entre conjuntos.
assert "id_caballo" in df_alineados.columns, "se perdió la trazabilidad por caballo"
print("\nCaballos representados:", df_alineados["id_caballo"].nunique())
print("Frames por clase:")
print(df_alineados["clase"].value_counts())

# Los frames alineados están en el disco local de la sesión: sin esto se pierden al cerrarla.
cache.empaquetar(alineados_dir, "frames_alineados")


## Bloque 5: Control de calidad

Paso 14 — Reporte de cobertura consolidado

Paso 15 — Inspección visual por clase

Paso 16 — Verificar que la alineación funciona

Paso 17 — Decisión

## Paso 14 — Reporte de cobertura consolidado

In [ ]:
# ============================================================
# REPORTE FINAL
# ============================================================

print("=" * 60)
print("COBERTURA DEL PREPROCESAMIENTO FACIAL")
print("=" * 60)

frames_esperados = len(df_clips) * NUM_FRAMES
print(f"Frames esperados:  {frames_esperados}")
print(f"Frames generados:  {len(df_alineados)} "
      f"({len(df_alineados)/frames_esperados:.1%})")
print(f"Frames alineados:  {int(df_alineados['alineado'].sum())} "
      f"({df_alineados['alineado'].mean():.1%})")

print("\nPor clase:")
resumen = df_alineados.groupby("clase").agg(
    frames=("alineado", "count"),
    alineados=("alineado", "sum"),
    tasa=("alineado", "mean"),
    caballos=("id_caballo", "nunique"),
)
print(resumen)

print("\nPor caballo:")
por_caballo = df_alineados.groupby("id_caballo").agg(
    frames=("alineado", "count"), tasa=("alineado", "mean")
).sort_values("tasa")
print(por_caballo)

problematicos = por_caballo[por_caballo["tasa"] < COBERTURA_MINIMA_ACEPTABLE]
if len(problematicos) > 0:
    print(f"\nADVERTENCIA: {len(problematicos)} caballos por debajo del "
          f"{COBERTURA_MINIMA_ACEPTABLE:.0%} de alineación.")
    print("No continuar al entrenamiento sin decidir qué hacer con ellos.")

# ============================================================
# PERSISTIR EL REPORTE
# ============================================================
# Este es el número que decide si el pipeline automático es utilizable o hay que anotar las 92
# ROI a mano. Imprimirlo no alcanza: la salida de una celda se pierde con la sesión.

reporte = por_caballo.reset_index()
reporte["clase"] = reporte["id_caballo"].map(
    df_alineados.groupby("id_caballo")["clase"].first()
)
reporte["por_debajo_del_umbral"] = reporte["tasa"] < COBERTURA_MINIMA_ACEPTABLE

reporte_path = resultados_dir / "reporte_cobertura.csv"
reporte.to_csv(reporte_path, index=False)

print("\nReporte de cobertura guardado en:", reporte_path)


## Paso 15 — Inspección visual por clase

In [ ]:
import matplotlib.pyplot as plt

# ============================================================
# GRILLA DE RECORTES ALINEADOS
# ============================================================
# Si la normalización funciona, dentro de cada fila las cabezas deben
# aparecer con la misma orientación y escala.

N_MUESTRAS = 6
clases = sorted(df_alineados["clase"].unique())

fig, axes = plt.subplots(len(clases), N_MUESTRAS,
                         figsize=(N_MUESTRAS * 2.2, len(clases) * 2.4))
axes = np.atleast_2d(axes)

for f, clase in enumerate(clases):
    disponibles = df_alineados[(df_alineados["clase"] == clase) &
                               (df_alineados["alineado"])]
    if len(disponibles) == 0:
        disponibles = df_alineados[df_alineados["clase"] == clase]

    muestra = disponibles.sample(min(N_MUESTRAS, len(disponibles)), random_state=42)

    for c in range(N_MUESTRAS):
        ax = axes[f, c]
        ax.axis("off")
        if c < len(muestra):
            img = cv2.imread(muestra.iloc[c]["ruta_frame"])
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            if c == 0:
                ax.set_title(clase, loc="left", fontsize=11)

plt.tight_layout()
plt.show()

## Paso 16 — Verificar que la alineación realmente funciona

Comprobación cuantitativa, no sólo visual: si la normalización es correcta, al proyectar los
keypoints originales con la misma transformada deben caer en la posición canónica en **todos** los
frames alineados. Es la propiedad que define el paso, y conviene medirla y no suponerla.

In [ ]:
# ============================================================
# VERIFICACIÓN NUMÉRICA DE LA ALINEACIÓN
# ============================================================

validos = df_det[df_det["par_valido"]]
errores = []

for _, fila in validos.iterrows():
    M = similitud_desde_dos_puntos(
        (fila["x_ojo"], fila["y_ojo"]),
        (fila["x_hocico"], fila["y_hocico"]),
        dest_ojo, dest_hocico,
    )
    if M is None:
        continue
    ojo = M[:, :2] @ np.array([fila["x_ojo"], fila["y_ojo"]]) + M[:, 2]
    hoc = M[:, :2] @ np.array([fila["x_hocico"], fila["y_hocico"]]) + M[:, 2]
    errores.append(max(np.hypot(*(ojo - dest_ojo)), np.hypot(*(hoc - dest_hocico))))

if errores:
    print(f"Frames verificados: {len(errores)}")
    print(f"Error máximo de alineación: {max(errores):.2e} píxeles")
    print(f"Error medio: {np.mean(errores):.2e} píxeles")
    assert max(errores) < 1e-6, "la alineación no lleva los puntos a la posición canónica"
    print("\nOK: en todos los frames el ojo y el hocico caen en la posición canónica.")
else:
    print("No hay frames con par válido para verificar.")

## Paso 17 — Decisión

Con los números del Paso 14 sobre la mesa:

**Si la tasa de alineación supera el umbral aceptable** → el objetivo específico 2 queda cumplido.
Continuar al Bloque 4 de modelado usando `dataset_frames_alineados.csv`, y entrenar también sobre
`dataset_frames.csv` (sin alinear) para poder reportar la comparación.

**Si no lo supera** → antes de seguir:

1. Revisar que la extracción del Paso 10 corresponda a la estructura real que devuelve el modelo.
   Es la causa más probable y la más barata de corregir.
2. Bajar `CONF_MINIMA_KEYPOINT` y volver a medir.
3. Si aun así no alcanza, anotar manualmente una ROI por clip. Son 92 cajas, unas pocas horas de
   trabajo, y produce una referencia exacta que además sirve para medir cualquier detector
   automático. **No es un fracaso del enfoque sino su red de contención.**

Anotar la cobertura obtenida en `openspec/project.md` en cualquiera de los dos casos: es un
resultado del trabajo, no un detalle de implementación.